# 2. Axis-angle

Run the cells from top to bottom. Select the `mano` kernel. The working directory may be `MANO` or `MANO/tutorial`.

Every MANO joint stores its rotation as 3 numbers. Those 3 numbers are an axis-angle.

## What they say

Treat the 3 numbers as a vector `r = (rx, ry, rz)`.

- The direction is the axis of rotation.
- The length is the angle, in radians.
- `(0, 0, 0)` has length 0, so it is no rotation.

`(π/2, 0, 0)` is 90 degrees about +X. `(0, 0, 0.7)` is 0.7 radians about +Z, about 40 degrees. Writing the same rotation as `(0, 0, 0.7 + 2π)` gives almost the same matrix, but the angles stored by MANO stay in the smaller range, and the training data sits there too.

Right-hand rule: point the right thumb along the positive axis. The curl of the fingers is the positive angle. Looking along +X, +Y turns toward +Z.

## Turning it into a matrix

Skinning needs a 3×3 rotation matrix. Rodrigues' formula does that. Let `θ = ||r||`. When `θ` is tiny, the matrix is the identity `I`. Otherwise let `k = r / θ` and write

```text
K = [[ 0,  -kz,  ky],
     [ kz,  0,  -kx],
     [-ky,  kx,  0 ]]

R = I + sin(θ) K + (1 - cos(θ)) K²
```

The next cell substitutes a 90-degree rotation about +X. Then `θ = π/2`, `k = (1, 0, 0)`, `sin θ = 1`, and `cos θ = 0`.


In [1]:
import sys
from pathlib import Path

import numpy as np

MANO = Path.cwd().resolve()
if not (MANO / "mano_skeleton.py").exists():
    MANO = MANO.parent
if str(MANO) not in sys.path:
    sys.path.insert(0, str(MANO))

def rodrigues(axis_angle):
    r = np.asarray(axis_angle, dtype=np.float64).reshape(3)
    theta = np.linalg.norm(r)
    if theta < 1e-12:
        return np.eye(3)
    k = r / theta
    kx, ky, kz = k
    K = np.array(
        [[0.0, -kz, ky], [kz, 0.0, -kx], [-ky, kx, 0.0]],
        dtype=np.float64,
    )
    return np.eye(3) + np.sin(theta) * K + (1.0 - np.cos(theta)) * (K @ K)

R = rodrigues([np.pi / 2, 0.0, 0.0])
print("90 degrees about +X:")
print(np.round(R, 5))
print("+Y goes to", np.round(R @ np.array([0.0, 1.0, 0.0]), 5))
print("zero axis-angle is the identity?", np.allclose(rodrigues([0, 0, 0]), np.eye(3)))

import mano_skeleton as sk
assert np.allclose(R, sk.rodrigues([np.pi / 2, 0.0, 0.0]))
assert np.allclose(R @ [0.0, 1.0, 0.0], [0.0, 0.0, 1.0])


90 degrees about +X:
[[ 1.  0.  0.]
 [ 0.  0. -1.]
 [ 0.  1.  0.]]
+Y goes to [0. 0. 1.]
zero axis-angle is the identity? True


You should see

```text
R = [[1, 0, 0],
     [0, 0,-1],
     [0, 1, 0]]
```

`(0, 1, 0)` becomes `(0, 0, 1)`. +Y has turned to +Z. That is the right-hand rule above. `mano_skeleton.rodrigues` is this same formula.

## How rotations stack

A point that is rotated by the child and then by the parent is

```text
R_global = R_parent @ R_local
```

`@` is matrix multiplication. The order does not commute. MANO's forward kinematics uses this order: the parent's global rotation is on the left, and the joint's own local rotation is on the right.


In [2]:
parent = rodrigues([0.0, 0.0, np.deg2rad(30)])
local = rodrigues([np.deg2rad(40), 0.0, 0.0])
stacked = parent @ local
print("local first, then parent:")
print(np.round(stacked, 3))
print("same matrix if you swap to local @ parent?", np.allclose(stacked, local @ parent))


local first, then parent:
[[ 0.866 -0.383  0.321]
 [ 0.5    0.663 -0.557]
 [ 0.     0.643  0.766]]
same matrix if you swap to local @ parent? False


In [4]:
parent, local, parent @ local, local @ parent

(array([[ 0.8660254, -0.5      ,  0.       ],
        [ 0.5      ,  0.8660254,  0.       ],
        [ 0.       ,  0.       ,  1.       ]]),
 array([[ 1.        ,  0.        ,  0.        ],
        [ 0.        ,  0.76604444, -0.64278761],
        [ 0.        ,  0.64278761,  0.76604444]]),
 array([[ 0.8660254 , -0.38302222,  0.3213938 ],
        [ 0.5       ,  0.66341395, -0.5566704 ],
        [ 0.        ,  0.64278761,  0.76604444]]),
 array([[ 0.8660254 , -0.5       ,  0.        ],
        [ 0.38302222,  0.66341395, -0.64278761],
        [ 0.3213938 ,  0.5566704 ,  0.76604444]]))

## Where the axes point at rest

At rest, each joint's local axes coincide with the model's X, Y, and Z. Local X does not automatically run along the bone.

Which of `rx`, `ry`, or `rz` flexes the index finger depends on where the fingers point in this template. The teaching sketch draws the four fingers along +Y, and in that drawing flexion is about +X. The official `MANO_RIGHT.pkl` points the four fingers along -X, so the flexion axis is a different one. Fill in the numbers after chapter 10 has measured them. Until then, an axis-angle is only "an axis plus a radian angle".

## Check

Cover the matrix and change the angle. `(π/2, 0, 0)` must send +Y to +Z. `(0, 0, 0)` must be the identity.

The next chapter lines the 16 joints up with these 48 numbers.
